# Hamiltonian Monte Carlo Sampling (in Tensorflow)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow_probability.substrates import jax as tfp

import matplotlib.pyplot as plt
import timeit

In [ ]:
tf.config.list_physical_devices('GPU')

## Sampling from diagonal-variance Gaussian

In [ ]:
tfd = tfp.distributions

dims = 10
true_stddev = tf.sqrt(tf.linspace(1., 3., dims))
likelihood = tfd.MultivariateNormalDiag(loc=0., scale_diag=true_stddev)

states = tfp.mcmc.sample_chain(
    num_results=1,
    num_burnin_steps=500,
    current_state=tf.zeros(dims),
    kernel=tfp.mcmc.HamiltonianMonteCarlo(
      target_log_prob_fn=likelihood.log_prob,
      step_size=0.5,
      num_leapfrog_steps=2),
    trace_fn=None)

sample_mean = tf.reduce_mean(states, axis=0)
# ==> approx all zeros

sample_stddev = tf.sqrt(tf.reduce_mean(
    tf.math.squared_difference(states, sample_mean),
    axis=0))
# ==> approx equal true_stddev

In [ ]:
print(np.mean(sample_mean), np.mean(sample_stddev))

## Simple chain with warm-up
In this example we sample from a standard univariate normal distribution using HMC with adaptive step size.

In [3]:
# Target distribution is proportional to: `exp(argument)`.
def unnormalized_log_prob(x):
  return -x - x**2.

# Initialize the HMC transition kernel.
num_results = int(100)
num_burnin_steps = int(1e3)
adaptive_hmc = tfp.mcmc.SimpleStepSizeAdaptation(
    tfp.mcmc.HamiltonianMonteCarlo(
        target_log_prob_fn=unnormalized_log_prob,
        num_leapfrog_steps=3,
        step_size=1.),
    num_adaptation_steps=int(num_burnin_steps * 0.8))

# Run the chain (with burn-in).
@tf.function
def run_chain():
  # Run the chain (with burn-in).
  samples, is_accepted = tfp.mcmc.sample_chain(
      num_results=num_results,
      num_burnin_steps=num_burnin_steps,
      current_state=1.,
      kernel=adaptive_hmc,
      trace_fn=lambda _, pkr: pkr.inner_results.is_accepted)

  sample_mean = tf.reduce_mean(samples)
  sample_stddev = tf.math.reduce_std(samples)
  is_accepted = tf.reduce_mean(tf.cast(is_accepted, dtype=tf.float32))
  return samples, sample_mean, sample_stddev, is_accepted

samples, sample_mean, sample_stddev, is_accepted = run_chain()

print('mean:{:.4f}  stddev:{:.4f}  acceptance:{:.4f}'.format(
    sample_mean.numpy(), sample_stddev.numpy(), is_accepted.numpy()))

mean:-0.5435  stddev:0.8017  acceptance:0.6300


In [19]:
print(states)

tf.Tensor(
[[ 0.02992213  0.21387273  1.0959024  ...  0.0707103   0.04861522
  -1.5157819 ]
 [-1.2441792   0.1991989   0.55753607 ...  0.24295568 -1.4158287
  -0.4341082 ]
 [-1.2124608  -0.74194    -0.61337775 ...  0.15612966  0.56778103
  -0.4032921 ]
 ...
 [ 2.2436907   1.7847115  -0.06776717 ... -0.5695236   0.08951032
   1.0620916 ]
 [ 1.1563568   1.5940912   1.1920326  ... -0.1634755   1.6988795
   0.7084296 ]
 [ 0.07280493  1.8470333   1.4800745  ...  0.95931023  1.8534498
  -1.2239392 ]], shape=(1000, 10), dtype=float32)


## Complex $\phi^4 $ theory

In this tutorial, we will simply go over how to use recently released TensorFlow Probability package to do Hamiltonian Monte Carlo(HMC) sampling. It makes doing HMC easier for everybody. First, you need to install TensorFlow and TensorFlow_probability if you haven't done so. The task we want to achieve is to get sufficient amount of samples $\phi$from a given probability distribution:
$\begin{equation*}
P_{target}=\frac{\exp(−S[\phi])}{\mathcal{Z}}
\end{equation*}$

,where $\mathcal{Z}$ is the partition function, and in physics, we call $S[\phi]$ the action of field $\phi$. If you are not familiar with physics, it is fine, and this tutorial is not about physics. It is just the notations. And you should notice that any probability distribution can be written into this form. $\phi=( \phi_0,\phi_1...)$ is a multi-dimensional vector, or a field configuration. From now on, we will use $\phi$ to implicitly denote a vector. 
The only difference between HMC and traditional MCMC is about how to update one configuration to get the next one. In HMC, we view $S[\phi]$ as a potential for $\phi$. In analog to classical mechanics, we can add a kinetic energy term: $K(\pi)=12m\pi^2$ to write down the total energy or Hamiltonian:
$\begin{equation*}
H=K(\pi)+S[\phi]
\end{equation*}$

Usually, we will set $m=1$. This is just to fix the total energy scale, and TensorFlow implicitly encode this in the package. In order to get an updated configuration, we evolve this configuration in the phase space $(\phi,\pi)$ using Hamiltonian equation:
$\begin{equation*}
\frac{d\phi}{dt}=\frac{\partial H}{\partial \pi},\frac{d\pi}{dt}=-\frac{\partial H}{\partial \phi}
\end{equation*}$

We let the system evolve a small amount of time $\delta t$, to get the updated configuration $\phi'$. Then the accept/reject procedure is the same as traditional MCMC. In the following, I will demonstrate how to use TensorFlow Probability to do sample on complex $\phi^4$ theory. If you are not familiar with the theory, the essential part is the probability density of $\phi$ is given by:
​$\begin{equation*}
S[\phi]=−k\sum_{<i,j>}\phi_i^∗\phi_j+\sum_{i}m|\phi_i|^2+\sum_{i}λ|\phi_i|^4
\end{equation*}$

where $\phi_i$ is a complex field lives on two dimensional lattice, and $<i,j>$ denotes the i-th and j-th locations are the nearest neighbor. 

We define our two dimensional lattice complex $\phi^4$ theory, and its parameters. (The parameters are chosen in order to create a deep Mexican hat, and the model is reduced to XY model controlled by temperature T). This part is not essential for those who are not interested in physics, and is safe to skip.

In [4]:
class phi4_class():
    def __init__(self, k, m, lam):
        self.k = k
        self.m = m
        self.lam = lam
    def energy(self, phi):
        '''
        dim(phi) = [batch, x, y, channel]
        '''
        energy = tf.reduce_sum(-self.k*(phi * tf.roll(phi, 1, axis=1) + 
                         phi * tf.roll(phi, 1, axis=2)),axis=[1,2,3] )
        phi_sqr = tf.reduce_sum(phi**2, axis=[3])
        energy += self.m * tf.reduce_sum(phi_sqr, axis=[1,2])
        energy += self.lam * tf.reduce_sum(phi_sqr**2, axis = [1,2])
        return energy
    def log_prob(self, phi):
        '''
        dim(phi) = [batch, x, y, channel]
        '''
        energy = tf.reduce_sum(-self.k*(phi * tf.roll(phi, 1, axis=1) + 
                         phi * tf.roll(phi, 1, axis=2)),axis=[1,2,3] )
        phi_sqr = tf.reduce_sum(phi**2, axis=[3])
        energy += self.m * tf.reduce_sum(phi_sqr, axis=[1,2])
        energy += self.lam * tf.reduce_sum(phi_sqr**2, axis = [1,2])
        return -energy

In [5]:
T = 2.0
k = 1./(2*T)
r = -200.
m = 4*k + r
l = -r/8.
x_dim = 16
y_dim = 16
batch_dim = 20
channel_dim = 2
xy_system = phi4_class(k,m,l)

Now we define the target_log_prob that we want to sample from. Notice TensorFlow distribution will take x as dimension:[batch, vector dimension]

In [6]:
def target_log_prob(x):
    x_dim = 16
    y_dim = 16
    channel_dim = 2
    config = tf.reshape(x,(-1,x_dim, y_dim, channel_dim))
    return xy_system.log_prob(config)

The important part is to define the following HMC kernel.

In [7]:
num_results = int(1e4)
num_burnin_steps = int(1e5)
adaptive_hmc = tfp.mcmc.SimpleStepSizeAdaptation(
    tfp.mcmc.HamiltonianMonteCarlo(
        target_log_prob_fn=target_log_prob,
        num_leapfrog_steps=20,
        step_size=1.),
    num_adaptation_steps=int(num_burnin_steps * 0.8))

In [8]:
@tf.function
def run_chain():
  # Run the chain (with burn-in).
    samples, is_accepted = tfp.mcmc.sample_chain(
        num_results=num_results,
        num_burnin_steps=num_burnin_steps,
        num_steps_between_results=int(20),
#        current_state=tf.convert_to_tensor(np.reshape(np_phi[:batch_dim,:,:,:],(-1,x_dim*y_dim*channel_dim)),dtype=DTYPE),
         current_state=tf.random.normal([batch_dim, x_dim*y_dim*channel_dim]),
        kernel=adaptive_hmc,
        trace_fn=lambda _, pkr: pkr.inner_results.is_accepted)

    sample_mean = tf.reduce_mean(samples)
    sample_stddev = tf.math.reduce_std(samples)
    is_accepted = tf.reduce_mean(tf.cast(is_accepted, dtype=tf.float32))
    return samples, sample_mean, sample_stddev, is_accepted

Now we can do the HMC sampling from the defined the target_log_prob.

In [9]:
start = timeit.default_timer()
samples, sample_mean, sample_stddev, is_accepted = run_chain()
stop = timeit.default_timer()
print('time:',np.round((stop-start)/60,2),"minutes")

time: 16.07 minutes


In [16]:
print(samples[0])

tf.Tensor(
[[ 1.7514216   0.92270404  1.8251313  ... -1.4954215   1.771831
  -0.8269278 ]
 [ 0.56375647  1.9547026   0.131345   ...  1.3990502   0.30339998
   1.954936  ]
 [-1.3307871  -1.4257672  -0.13684437 ... -0.16291018  0.92269415
  -1.8357621 ]
 ...
 [ 0.66553205 -1.8534434   1.9910179  ... -1.5402904   1.8163657
  -0.67409337]
 [ 1.168575   -1.606531    1.3358916  ... -1.9847153   1.4527516
  -1.316231  ]
 [-0.38146564  1.9873614  -0.1443101  ...  1.0835428  -0.20380223
   1.9835026 ]], shape=(20, 512), dtype=float32)
